# Jev research v0 — hosted GPU (Google Colab)

This notebook is the runnable form of [`docs/11-colab.md`](https://github.com/WiredMind2/jev/blob/main/docs/11-colab.md).
It is **not** TypeSafe Jev, **not** RLCD, and **not** a serving runtime.

| This box | Use it for |
|---|---|
| Free Colab T4 (~16 GiB) | Frozen-head train/eval for 0.5B–3B that does not fit a 4 GiB GTX 1650 |
| Local CPU | Synthetic hashing smoke only (`scripts/colab_cpu_smoke.sh`) |
| `jev serve` | **Do not run** here. Ephemeral VM, idle disconnect. |

The v0 table in `reports/v0/metrics.md` stays on the 1650 / Qwen2.5-0.5B pin.
Colab numbers need a **new** hardware note. Cells only provision the runtime
and invoke `python -m jev`.

**Runtime:** Runtime → Change runtime type → T4 GPU. Python ≥ 3.11.

## 1. GPU check

Stop a Colab GPU job if CUDA is missing. `jev hardware` prints the **repo pin**
(GTX 1650 / 0.5B), not the live accelerator — record `nvidia-smi` and
`torch.cuda.get_device_properties` into the run note.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import torch

print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, p.total_memory // (1024**2), "MiB")
    subprocess.run(["nvidia-smi"], check=False)
elif IN_COLAB:
    raise SystemExit("Colab runtime has no CUDA. Runtime → Change runtime type → T4 GPU.")
else:
    print("No CUDA: synthetic hashing smoke only. Do not train an HF encoder on CPU for the hosted-GPU jobs.")
if torch.cuda.is_available() and not IN_COLAB:
    print("Local CUDA is not the v0 measurement box. Hosted-GPU Qwen jobs run on a Colab T4, not a laptop GPU.")


## 2. Clone and install without clobbering Colab's CUDA torch

Colab already ships a CUDA PyTorch. A naive `pip install -e ".[dev]"` can
replace it with a CPU wheel. Install the package with `--no-deps`, then the
other requirements, then re-check CUDA.

In [ ]:
REPO = os.environ.get("JEV_REPO", "https://github.com/WiredMind2/jev.git")
REPO_REF = os.environ.get("JEV_REF", "")  # pin a commit/branch when reproducing a report

def _in_clone() -> bool:
    return Path("pyproject.toml").exists() and Path("src/jev").exists()

if _in_clone():
    print("already in clone", Path(".").resolve())
elif (Path("jev") / "pyproject.toml").exists():
    os.chdir("jev")
else:
    cmd = ["git", "clone", "--depth", "1", REPO, "jev"]
    if REPO_REF:
        cmd = ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO, "jev"]
    subprocess.check_call(cmd)
    os.chdir("jev")
    if REPO_REF and REPO_REF not in subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True):
        subprocess.check_call(["git", "fetch", "--depth", "1", "origin", REPO_REF])
        subprocess.check_call(["git", "checkout", REPO_REF])

_target = REPO_REF or "main"
subprocess.check_call(["git", "fetch", "--depth", "1", "origin", _target])
subprocess.check_call(["git", "checkout", "-q", "FETCH_HEAD"])
print("cwd", Path(".").resolve())
print("git HEAD", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[dev]", "--no-deps"])
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "pydantic>=2.7",
        "jsonschema>=4.22",
        "fastapi>=0.111",
        "uvicorn[standard]>=0.30",
        "typer>=0.12",
        "numpy>=1.26",
        "scikit-learn>=1.4",
        "transformers>=4.44",
        "datasets>=2.20",
        "safetensors>=0.4",
        "httpx>=0.27",
        "pyyaml>=6.0",
        "pytest>=8.2",
    ]
)
import torch as _torch
if IN_COLAB:
    assert _torch.cuda.is_available(), "CUDA torch missing after pip; do not train"
print("cuda after install", _torch.cuda.is_available(), _torch.__version__)

## 3. Optional Hugging Face token (Colab Secrets only)

Never paste a token into a cell or commit one. If a gated download needs auth,
add Colab Secrets key `HF_TOKEN`. Qwen2.5-0.5B is typically ungated.
`data-convert banking77` may still fail on `PolyAI/banking77` unauthenticated;
the converter already tries `mteb/banking77`. Do not vendor the raw corpus.

In [ ]:
token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if IN_COLAB and not token:
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        print("no Colab secret HF_TOKEN:", type(exc).__name__)
        token = None
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN set from environment or Colab Secrets (value not printed)")
else:
    print("no HF_TOKEN; ungated downloads only")

## 4. Persist artifacts (same layout as `data/README.md`)

Disk under `/content` disappears when the runtime dies. Put checkpoints,
converted JSONL, and eval JSON on Drive (or a local `JEV_RUNS` directory)
**before** the job finishes.

In [ ]:
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/jev-runs")
else:
    ROOT = Path(os.environ.get("JEV_RUNS", str(Path.cwd() / ".jev-colab-smoke"))).resolve()

for name in ("data", "runs", "reports"):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
(ROOT / "reports" / "metrics").mkdir(parents=True, exist_ok=True)
(ROOT / "reports" / "model-cards").mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(ROOT / "hf-cache")
DATA, RUNS, REPORTS = ROOT / "data", ROOT / "runs", ROOT / "reports"
print("ROOT", ROOT)
print("HF_HOME", os.environ["HF_HOME"])


def run_jev(*args: str) -> None:
    cmd = [sys.executable, "-m", "jev", *args]
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


SLICE = {
    "banking77": 300,
    "sst5": 300,
    "boolq": 300,
    "wikispeedia": 300,
    "clinc150": 300,
}
MAX_STEPS = {
    "banking77": "400",
    "sst5": "400",
    "boolq": "400",
    "wikispeedia": "400",
    "clinc150": "800",
}


def run_frozen_split_job(
    name: str,
    *,
    batch_size: int = 4,
    epochs: str = "12",
    model_id: str = "Qwen/Qwen2.5-0.5B",
) -> None:
    """Convert, train (resume), calibrate, evaluate via python -m jev only."""
    run_jev("data-convert", name, "--out", str(DATA))
    jsonl = DATA / name / "jsonl"
    train = jsonl / "train.jsonl"
    val = jsonl / "validation.jsonl"
    calib = jsonl / "calibration.jsonl"
    test = jsonl / "test.jsonl"
    ckpt = RUNS / f"{name}-hf-head.pt"
    t_head = RUNS / f"{name}-head-temperature.json"
    t_zs = RUNS / f"{name}-zeroshot-temperature.json"
    limit = str(SLICE[name])
    train_args = [
        "train-head",
        str(train),
        "--val-jsonl",
        str(val),
        "--encoder",
        "hf",
        "--model-id",
        model_id,
        "--out",
        str(ckpt),
        "--epochs",
        epochs,
        "--batch-size",
        str(batch_size),
        "--save-every",
        "50",
    ]
    steps = os.environ.get(f"JEV_MAX_STEPS_{name.upper()}") or MAX_STEPS.get(name)
    if steps:
        train_args.extend(["--max-steps", str(steps)])
        print(name, "train-head --max-steps", steps, "(resume across Colab sessions)")
    if ckpt.exists():
        train_args.extend(["--resume", str(ckpt)])
    run_jev(*train_args)
    from jev.hardware import release_cuda

    release_cuda()
    run_jev(
        "calibrate",
        str(calib),
        "--backend",
        "option-head",
        "--checkpoint",
        str(ckpt),
        "--limit",
        limit,
        "--seed",
        "0",
        "--out",
        str(t_head),
    )
    run_jev(
        "evaluate",
        str(test),
        "--backend",
        "option-head",
        "--checkpoint",
        str(ckpt),
        "--temperature-json",
        str(t_head),
        "--limit",
        limit,
        "--seed",
        "0",
        "--out",
        str(REPORTS / "metrics" / f"eval-{name}-hf-head.json"),
    )
    release_cuda()
    run_jev(
        "calibrate",
        str(calib),
        "--backend",
        "hf-logprob",
        "--limit",
        limit,
        "--seed",
        "0",
        "--out",
        str(t_zs),
    )
    run_jev(
        "evaluate",
        str(test),
        "--backend",
        "hf-logprob",
        "--temperature-json",
        str(t_zs),
        "--limit",
        limit,
        "--seed",
        "0",
        "--out",
        str(REPORTS / "metrics" / f"eval-{name}-hf-logprob.json"),
    )
    run_jev(
        "evaluate",
        str(test),
        "--backend",
        "majority",
        "--train-jsonl",
        str(train),
        "--limit",
        limit,
        "--seed",
        "0",
        "--out",
        str(REPORTS / "metrics" / f"eval-{name}-majority.json"),
    )
    if name == "wikispeedia":
        print("skip tfidf-linear for wikispeedia (dense multinomial does not fit Colab RAM)")
    else:
        run_jev(
            "evaluate",
            str(test),
            "--backend",
            "tfidf-linear",
            "--train-jsonl",
            str(train),
            "--limit",
            limit,
            "--seed",
            "0",
            "--out",
            str(REPORTS / "metrics" / f"eval-{name}-tfidf.json"),
        )
    print("wrote evals for", name, "limit", limit, "under", REPORTS / "metrics")

## 5. Record live GPU vs repo pin

`python -m jev hardware` is the 1650 pin on purpose. Write a run-specific note
instead of editing `configs/hardware.yaml` or `reports/v0/hardware.md`.

In [ ]:
run_jev("hardware")
note = {
    "source": "notebooks/jev_colab_hosted_gpu.ipynb",
    "in_colab": IN_COLAB,
    "torch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "live_gpu": None,
    "repo_pin_command": "python -m jev hardware",
    "do_not_mix_with": "reports/v0/metrics.md",
}
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    note["live_gpu"] = {
        "name": p.name,
        "total_mib": int(p.total_memory // (1024**2)),
        "index": 0,
    }
path = REPORTS / "colab-live-hardware.json"
path.write_text(json.dumps(note, indent=2) + "\n", encoding="utf-8")
md = REPORTS / "hardware.md"
gpu = note.get("live_gpu") or {}
md.write_text(
    "\n".join(
        [
            "# Colab live GPU (not the 1650 pin)",
            "",
            "Do not copy these rows into reports/v0/metrics.md.",
            "",
            f"- in_colab: {IN_COLAB}",
            f"- torch: {torch.__version__}",
            f"- cuda_available: {bool(torch.cuda.is_available())}",
            f"- name: {gpu.get('name', 'none')}",
            f"- total_mib: {gpu.get('total_mib', 0)}",
            "- repo pin remains GTX 1650 / Qwen2.5-0.5B (`python -m jev hardware`)",
            "",
        ]
    ),
    encoding="utf-8",
)
print("wrote", path)
print("wrote", md)

## 6. Synthetic CLI smoke (hashing head)

Proves convert → train-head → calibrate → evaluate. This is the CPU-capable
path. It is **not** a replacement for the v0 0.5B synthetic frozen-head cells
already in `reports/v0/`. Never pass the calibration split to `--val-jsonl`.

In [ ]:
import json

SMOKE_N = int(os.environ.get("SYNTHETIC_N", "256" if IN_COLAB and torch.cuda.is_available() else "128"))
run_jev("data-convert", "synthetic", "--out", str(DATA), "--n", str(SMOKE_N))

train = DATA / "synthetic" / "jsonl" / "train.jsonl"
val = DATA / "synthetic" / "jsonl" / "validation.jsonl"
calib = DATA / "synthetic" / "jsonl" / "calibration.jsonl"
test = DATA / "synthetic" / "jsonl" / "test.jsonl"
ckpt = RUNS / "synthetic-hashing-head.pt"
cal_json = RUNS / "synthetic-hashing-temperature.json"
eval_json = REPORTS / "eval-synthetic-hashing.json"

run_jev(
    "train-head", str(train),
    "--val-jsonl", str(val),
    "--encoder", "hashing",
    "--out", str(ckpt),
    "--epochs", "12",
    "--batch-size", "16",
)
run_jev("calibrate", str(calib), "--checkpoint", str(ckpt), "--out", str(cal_json))
run_jev(
    "evaluate", str(test),
    "--backend", "option-head",
    "--checkpoint", str(ckpt),
    "--temperature-json", str(cal_json),
    "--out", str(eval_json),
)
print(eval_json.read_text(encoding="utf-8"))

## 7. Tiny CUDA gates (optional)

Useful if you have no local NVIDIA GPU. Full HF CUDA gates download Qwen and
need VRAM; run them only after `HF_HOME` is on Drive.

In [ ]:
if torch.cuda.is_available():
    subprocess.check_call([sys.executable, "-m", "pytest", "-m", "cuda and not hf", "-q"])
else:
    print("skip pytest -m cuda (no GPU)")

## 8. Hosted-GPU job: BANKING77 frozen head vs zero-shot (0.5B)

This is the recipe in `docs/11-colab.md`. Enable it on a T4. Keep the encoder
frozen. Batch size 4 is a starting point; drop it if VRAM is tight.
`--resume` continues from Drive if the VM died. Comparison slice is a pinned
stratified `--limit 300` of the frozen official test (seed 0).

Set `RUN_BANKING77=1` or leave the default: on when Colab+CUDA, off on CPU.

In [ ]:
RUN_BANKING77 = os.environ.get("RUN_BANKING77")
if RUN_BANKING77 is None:
    RUN_BANKING77 = "1" if (IN_COLAB and torch.cuda.is_available()) else "0"
if RUN_BANKING77 != "1":
    print("skip BANKING77 hosted-GPU job (set RUN_BANKING77=1 on a T4)")
else:
    run_frozen_split_job("banking77", batch_size=4, epochs="12")

## 8b. SST-5 Score (frozen split)

Same CLI helper as BANKING77. Official test preferred; calibration carved from train.
Set `RUN_SST5=1` (default on Colab+CUDA after you want the next job).

In [ ]:
RUN_SST5 = os.environ.get("RUN_SST5")
if RUN_SST5 is None:
    RUN_SST5 = "1" if (IN_COLAB and torch.cuda.is_available()) else "0"
if RUN_SST5 != "1":
    print("skip SST-5 hosted-GPU job (set RUN_SST5=1 on a T4)")
else:
    run_frozen_split_job("sst5", batch_size=4, epochs="12")

## 8c. BoolQ Noul (frozen split)

Page-grouped split. Converter falls back to `google/boolq` if the original JSONL is 403.
Set `RUN_BOOLQ=1`.

In [ ]:
RUN_BOOLQ = os.environ.get("RUN_BOOLQ")
if RUN_BOOLQ is None:
    RUN_BOOLQ = "1" if (IN_COLAB and torch.cuda.is_available()) else "0"
if RUN_BOOLQ != "1":
    print("skip BoolQ hosted-GPU job (set RUN_BOOLQ=1 on a T4)")
else:
    run_frozen_split_job("boolq", batch_size=2, epochs="12")

## 8d. Wikispeedia next-click (variable Choice)

Target-disjoint split, variable N. Train may need `--resume` across sessions.
Shuffled-context should collapse toward chance. Set `RUN_WIKISPEEDIA=1`.

In [ ]:
RUN_WIKISPEEDIA = os.environ.get("RUN_WIKISPEEDIA")
if RUN_WIKISPEEDIA is None:
    RUN_WIKISPEEDIA = "1" if (IN_COLAB and torch.cuda.is_available()) else "0"
if RUN_WIKISPEEDIA != "1":
    print("skip Wikispeedia hosted-GPU job (set RUN_WIKISPEEDIA=1 on a T4)")
else:
    run_frozen_split_job("wikispeedia", batch_size=1, epochs="6")

## 8e. CLINC150 + out_of_scope (then)

151-way Choice including `out_of_scope`. Run after the first four public tasks.
Set `RUN_CLINC150=1`.

In [ ]:
RUN_CLINC150 = os.environ.get("RUN_CLINC150")
if RUN_CLINC150 is None:
    RUN_CLINC150 = "1" if (IN_COLAB and torch.cuda.is_available()) else "0"
if RUN_CLINC150 != "1":
    print("skip CLINC150 hosted-GPU job (set RUN_CLINC150=1 on a T4)")
else:
    run_frozen_split_job("clinc150", batch_size=2, epochs="12")

## 9. Optional 3B VRAM probe (T4 only)

The 1650 pin exists because 3B fp16 weights exceed 4 GiB. On a T4: keep the
encoder frozen, start from the same JSONL as 0.5B, then `--model-id Qwen/Qwen2.5-3B`
with `batch_size` 1 and a short `--max-steps` probe. Do not jump to 7B/8B fp16
on free Colab. Do not edit the v0 1650 pin to match this box.

In [ ]:
RUN_3B_PROBE = os.environ.get("RUN_3B_PROBE")
if RUN_3B_PROBE is None:
    RUN_3B_PROBE = "1" if (IN_COLAB and torch.cuda.is_available()) else "0"
if RUN_3B_PROBE != "1":
    print("skip 3B probe (set RUN_3B_PROBE=1 after 0.5B BANKING77 succeeds)")
elif not torch.cuda.is_available():
    raise SystemExit("3B probe needs CUDA")
else:
    from jev.hardware import release_cuda
    release_cuda()
    probe_train = DATA / "synthetic" / "jsonl" / "train.jsonl"
    probe_ckpt = RUNS / "synthetic-qwen25-3b-probe.pt"
    outcome = "fit (2 steps completed)"
    try:
        run_jev(
            "train-head", str(probe_train),
            "--encoder", "hf",
            "--model-id", "Qwen/Qwen2.5-3B",
            "--out", str(probe_ckpt),
            "--epochs", "1",
            "--batch-size", "1",
            "--max-steps", "2",
        )
    except subprocess.CalledProcessError as exc:
        outcome = f"failed exit={exc.returncode} (likely OOM or load error)"
    card = REPORTS / "model-cards" / "qwen25-3b-probe.md"
    card.write_text(
        "# Model card — Qwen2.5-3B VRAM probe\n\n"
        f"**Outcome:** {outcome}\n\n"
        "Command: `jev train-head --encoder hf --model-id Qwen/Qwen2.5-3B --batch-size 1 --max-steps 2`.\n"
        "Encoder stays frozen. File under reports/colab-t4/; do not mix into reports/v0.\n",
        encoding="utf-8",
    )
    print("3B max-steps probe", outcome, "wrote", card)
    release_cuda()

## What to check into git

- Converted datasets stay out of git (`docs/10-datasets.md`).
- Checkpoints (`.pt`) stay out of git unless a model card vendors a tiny proof artifact.
- Eval JSON, a hardware note, and a model card **do** belong in `reports/` for a
  **named Colab run**, with the live GPU recorded. Do not paste into
  `reports/v0/metrics.md`.
- This notebook and `docs/11-colab.md`. Not a claim that Colab numbers equal
  hosted Jev or the v0 1650 table.

Related: `configs/hardware.yaml`, `reports/v0/hardware.md`, `reports/v0/colab.md`,
`docs/04-training.md`, `docs/06-implementation.md` (week 3 is the Colab week).